# 고객 이탈(`churn.csv`) 탐색적 데이터 분석(EDA)

동일 디렉터리에 `churn.csv`를 두고 실행합니다. (Google Drive 등에서 받은 파일을 프로젝트 루트에 복사하세요.)

**목표**: 데이터 품질 점검, 단변량·이변량·다변량 탐색, 이탈과 연관된 패턴 요약.

## 1. 환경 설정 및 데이터 로딩

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["font.size"] = 11

DATA_PATH = Path("churn.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"'{DATA_PATH.resolve()}' 가 없습니다. churn.csv를 이 노트북과 같은 폴더에 두세요."
    )

df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
df.head()

In [ ]:
df.info()
df.describe(include="all").T

In [ ]:
dup_rows = df.duplicated().sum()
print("duplicate rows:", dup_rows)

id_cols = [c for c in df.columns if str(c).lower() in ("customerid", "customer_id", "id")]
if id_cols:
    print("id duplicate counts:", {c: df[c].duplicated().sum() for c in id_cols})

### 타깃 컬럼(`Churn`) 지정

이름이 `Churn`/`churn`/`CHURN` 등이면 자동 선택하고, 없으면 아래 `TARGET_COL`을 수동으로 바꿉니다.

In [ ]:
def infer_target_column(columns):
    for name in columns:
        if str(name).strip().lower() == "churn":
            return name
    raise ValueError("타깃 컬럼을 찾지 못했습니다. TARGET_COL을 직접 지정하세요.")


TARGET_COL = infer_target_column(df.columns)
print("TARGET_COL =", TARGET_COL)

y_raw = df[TARGET_COL]
if y_raw.dtype == object:
    y_norm = y_raw.astype(str).str.strip().str.lower()
    pos = y_norm.isin({"yes", "y", "true", "1", "churn", "이탈"})
    neg = y_norm.isin({"no", "n", "false", "0", "stay", "비이탈"})
    if pos.any() and neg.any():
        y = pos.astype(int)
    else:
        y = pd.to_numeric(y_raw, errors="coerce")
else:
    y = pd.to_numeric(y_raw, errors="coerce")

df = df.assign(_churn_binary=y)
print(df[TARGET_COL].value_counts(dropna=False))
print("binary (0/1) value counts:")
print(df["_churn_binary"].value_counts(dropna=False))

## 2. 데이터 품질 진단 (결측·이상·범주 정규화)

In [ ]:
na_rate = df.isna().mean().sort_values(ascending=False)
na_rate[na_rate > 0]

In [ ]:
def strip_object_columns(frame):
    out = frame.copy()
    for c in out.select_dtypes(include="object").columns:
        if c == TARGET_COL:
            continue
        out[c] = out[c].astype(str).str.strip()
        out[c] = out[c].replace({"nan": np.nan})
    return out


df = strip_object_columns(df)
feature_cols = [c for c in df.columns if c not in (TARGET_COL, "_churn_binary")]
num_cols = df[feature_cols].select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]
print("numeric:", num_cols)
print("categorical:", cat_cols)

In [ ]:
for c in num_cols:
    s = df[c]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out = ((s < low) | (s > high)).sum()
    if out:
        print(f"{c}: IQR 기준 이상치 후보 {out}건 (참고용, 도메인에 맞게 해석)")

## 3. 단변량 분포

In [ ]:
for c in num_cols[:12]:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    sns.histplot(df[c].dropna(), kde=True, ax=ax[0])
    ax[0].set_title(f"{c} histogram")
    sns.boxplot(x=df[c], ax=ax[1])
    ax[1].set_title(f"{c} boxplot")
    plt.tight_layout()
    plt.show()

if len(num_cols) > 12:
    print("(수치형이 많아 처음 12개만 그렸습니다. 필요 시 슬라이스를 조정하세요.)")

In [ ]:
for c in cat_cols[:15]:
    vc = df[c].value_counts(dropna=False)
    plt.figure(figsize=(8, 3))
    sns.barplot(x=vc.values, y=vc.index.astype(str), orient="h")
    plt.title(f"{c} frequency")
    plt.tight_layout()
    plt.show()

if len(cat_cols) > 15:
    print("(범주형이 많아 처음 15개만 그렸습니다.)")

## 4. 타깃 중심 이변량 분석

In [ ]:
valid = df["_churn_binary"].notna()
df_valid = df.loc[valid].copy()
print("rows with valid target:", len(df_valid), "/", len(df))

In [ ]:
for c in num_cols[:12]:
    plt.figure(figsize=(8, 3))
    sns.boxplot(data=df_valid, x="_churn_binary", y=c)
    plt.title(f"{c} by churn (0/1)")
    plt.tight_layout()
    plt.show()

In [ ]:
for c in cat_cols[:12]:
    ct = pd.crosstab(df_valid[c], df_valid["_churn_binary"], normalize="index")
    display(ct.head(20))
    plt.figure(figsize=(8, 3))
    churn_rate = df_valid.groupby(c, dropna=False)["_churn_binary"].mean().sort_values(ascending=False)
    sns.barplot(x=churn_rate.values, y=churn_rate.index.astype(str), orient="h")
    plt.title(f"Churn rate by {c}")
    plt.xlabel("mean(_churn_binary)")
    plt.tight_layout()
    plt.show()

## 5. 다변량 관계 (상관·페어)

In [ ]:
if len(num_cols) >= 2:
    corr = df_valid[num_cols].corr(numeric_only=True)
    plt.figure(figsize=(max(8, len(num_cols) * 0.4), max(6, len(num_cols) * 0.35)))
    sns.heatmap(corr, annot=len(num_cols) <= 12, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Numeric correlation")
    plt.tight_layout()
    plt.show()
else:
    print("수치형 컬럼이 2개 미만이라 상관 히트맵을 생략합니다.")

In [ ]:
pair_cols = [c for c in num_cols if df_valid[c].notna().sum() > 10][:4]
if len(pair_cols) >= 2:
    sns.pairplot(
        df_valid[pair_cols + ["_churn_binary"]].dropna(),
        hue="_churn_binary",
        corner=True,
        plot_kws={"alpha": 0.35, "s": 12},
    )
    plt.suptitle("Pairplot (subset of numeric)", y=1.02)
    plt.show()
else:
    print("페어플롯에 쓸 수치형이 부족합니다.")

## 6. 요약 (인사이트 메모)

아래에 **이탈률이 높은 범주**, **이탈군에서 분포가 다른 수치형**, **상관이 큰 변수 쌍** 등을 직접 bullet로 정리하세요.

- 
- 
- 